In [2]:
import sys
sys.path.append("..")

from src.data import load_data, split
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

X_train, X_test, y_train, y_test = split(load_data())

xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print(f"Test RMSE: {root_mean_squared_error(y_test, y_pred):,.0f}")
print(f"Test R²:   {r2_score(y_test, y_pred):.3f}")

Test RMSE: 10,141
Test R²:   0.873


In [3]:
y_train_pred = xgb.predict(X_train)
print(f"Train RMSE: {root_mean_squared_error(y_train, y_train_pred):,.0f}")

Train RMSE: 0


In [4]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid={
        "n_estimators": [50, 100],
        "max_depth": [2, 3],
        "learning_rate": [0.05, 0.1],
    },
    cv=5,
    scoring="neg_root_mean_squared_error",
)
grid.fit(X_train, y_train)

print("Best settings:", grid.best_params_)
print(f"Best CV RMSE: {-grid.best_score_:,.0f}")

Best settings: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best CV RMSE: 11,172


In [5]:
best_xgb = grid.best_estimator_
y_pred_best = best_xgb.predict(X_test)

print(f"Test RMSE: {root_mean_squared_error(y_test, y_pred_best):,.0f}")
print(f"Test R²:   {r2_score(y_test, y_pred_best):.3f}")

Test RMSE: 9,067
Test R²:   0.898


## Result
Linear Regression beat XGBoost: its CV RMSE was clearly lower (9,547 vs 11,172), and on the test set the two were almost tied (8,996 vs 9,067). With only 40 training rows and an almost straight-line relationship between R&D and Profit (correlation 0.97), a simple straight-line model is enough.

Default XGBoost overfitted: train RMSE was 0 (it memorised the training data) but test RMSE was 10,141.

Tuning helped by making the model simpler (shallow trees and a small learning rate), so it memorised less, and test RMSE dropped from 10,141 to 9,067.